In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline 

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import(mean_absolute_error,
    mean_squared_error,
    r2_score)

In [3]:
data = pd.read_csv("./data_folder/Feature_engineered_data.csv")

In [4]:
data.head(2)

,Order_Date,Year,Month,Quarter,Season,Customer_Gender,Customer_Segment,Region,Country,Category,...,Shipping_Days,Payment_Method,Order_Status,Order_Month,Order_Quarter,Order_Year,revenue_per_unit,profit_per_unit,scost_per_revenue,net_revenue
0,2021-01-05,2021,January,Q1,Winter,Female,New,Asia,India,Clothing,...,22,Cash on Delivery,Delivered,1,1,2021,56.095,-9.050,0.162849,6.140668
1,2021-01-15,2021,January,Q1,Winter,Female,Premium,Middle East,Jordan,Electronics,...,18,Credit Card,Delivered,1,1,2021,465.790,30.555,0.039857,25.089685


In [5]:
# Our target variable is profit

y = data["Profit"]

In [6]:
data.columns

Index(['Order_Date', 'Year', 'Month', 'Quarter', 'Season', 'Customer_Gender',
       'Customer_Segment', 'Region', 'Country', 'Category', 'Sub_Category',
       'Product_Name', 'Unit_Price', 'Quantity', 'Discount', 'Revenue', 'Cost',
       'Profit', 'Profit_Margin_%', 'Shipping_Cost', 'Shipping_Method',
       'Shipping_Days', 'Payment_Method', 'Order_Status', 'Order_Month',
       'Order_Quarter', 'Order_Year', 'revenue_per_unit', 'profit_per_unit',
       'scost_per_revenue', 'net_revenue'],
      dtype='object')

In [7]:
## dropping columns that directly reveal profit

data = data.drop(columns = ["Profit", "Quarter", "Month", "Order_Date", "Revenue", "Cost", "net_revenue", "revenue_per_unit", "profit_per_unit", "Profit_Margin_%" ])

In [8]:
data.head(2)

,Year,Season,Customer_Gender,Customer_Segment,Region,Country,Category,Sub_Category,Product_Name,Unit_Price,...,Discount,Shipping_Cost,Shipping_Method,Shipping_Days,Payment_Method,Order_Status,Order_Month,Order_Quarter,Order_Year,scost_per_revenue
0,2021,Winter,Female,New,Asia,India,Clothing,Men's Wear,Hoodie Sweatshirt,93.49,...,0.4,18.27,Standard,22,Cash on Delivery,Delivered,1,1,2021,0.162849
1,2021,Winter,Female,Premium,Middle East,Jordan,Electronics,Laptops,Lenovo ThinkPad X1,931.58,...,0.5,37.13,Express,18,Credit Card,Delivered,1,1,2021,0.039857


In [9]:
data.shape

(10000, 21)

In [10]:
x = data

In [11]:
## Identifying numerical and categorical columns

numerical_columns = x.select_dtypes(include=["int64", "float64"]).columns

categorical_columns = x.select_dtypes(include=["object"]).columns

In [12]:
numerical_columns

Index(['Year', 'Unit_Price', 'Quantity', 'Discount', 'Shipping_Cost',
       'Shipping_Days', 'Order_Month', 'Order_Quarter', 'Order_Year',
       'scost_per_revenue'],
      dtype='object')

In [13]:
categorical_columns

Index(['Season', 'Customer_Gender', 'Customer_Segment', 'Region', 'Country',
       'Category', 'Sub_Category', 'Product_Name', 'Shipping_Method',
       'Payment_Method', 'Order_Status'],
      dtype='object')

In [14]:
## Encoding categorical data

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown = "ignore"),
            categorical_columns
        )
    ],
    remainder="passthrough"
)

In [15]:
## splitting the data

x_train, x_test, y_train, y_test =train_test_split(
    x,
    y,
    test_size = 0.2,
    random_state = 34
)

In [16]:
## Training using MODEL 1(Linear regression)


First_pipeline = Pipeline([
    ("preprocessor", preprocessor),
("model", LinearRegression())])

In [17]:
First_pipeline.fit(x_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [18]:
first_pred = First_pipeline.predict(x_test)

In [19]:
first_pred

array([1215.79159154, -243.06759215,   82.62091251, ...,  170.92773639,
        214.54312995,  137.45212464], shape=(2000,))

In [20]:
## Evaluation

MAE = mean_absolute_error(y_test, first_pred)
MAE

150.43021223421442

In [21]:
MSE = np.sqrt(mean_squared_error(y_test, first_pred))
MSE

np.float64(342.7135332794928)

In [22]:
R2 = r2_score(y_test, first_pred)
R2

0.46221819795724317

In [23]:
## Model 2(Randomforestregressor)



rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=20,
        random_state = 34
    ))
])

In [24]:
rf_pipeline.fit(x_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:
rf_pred = rf_pipeline.predict(x_test)
rf_pred

array([248.0495, -15.493 ,  23.769 , ...,  98.627 ,  35.3915,  15.2295],
      shape=(2000,))

In [26]:
## evaluate


rf_MAE = mean_absolute_error(y_test, rf_pred)
print(rf_MAE)
rf_R2 = r2_score(y_test, rf_pred)
print(rf_R2)
rf_MSE = np.sqrt(mean_squared_error(y_test, rf_pred))
print(rf_MSE)

55.26455924999999
0.8814066683503188
160.9379030494522


In [27]:
## Model 3(Gradient Boosting)


gbpipeline = Pipeline([
    ("Preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        random_state = 34
    ))

])

In [28]:
gbpipeline.fit(x_train, y_train)

,steps,"[('Preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [29]:
gb_pred = gbpipeline.predict(x_test)
gb_pred

array([454.80204138, -50.45313246,  21.92962555, ..., 102.07699455,
        26.6553982 ,  19.18063317], shape=(2000,))

In [30]:
## evaluate


gb_MAE = mean_absolute_error(y_test, gb_pred)
print(gb_MAE)
gb_R2 = r2_score(y_test, gb_pred)
print(gb_R2)
gb_MSE = np.sqrt(mean_squared_error(y_test, gb_pred))
print(gb_MSE)

62.895946482704154
0.8689304667278148
169.1917100311156


In [31]:
# MAE,MSE,R2, rf_MAE,rf_MSE,rf_R2, gb_MAE,gb_MSE,gb_R2,

compare_model = pd.DataFrame([[MAE,MSE,R2], [rf_MAE,rf_MSE,rf_R2],[ gb_MAE,gb_MSE,gb_R2]],
                            index=["Linear Regression", "RandomForest Regressor", "Gradient Boosting"],
                            columns= ["MAE", "MSE", "R2"])
compare_model

,MAE,MSE,R2
Linear Regression,150.430212,342.713533,0.462218
RandomForest Regressor,55.264559,160.937903,0.881407
Gradient Boosting,62.895946,169.191710,0.868930


** RandomForestRegressor did better**

In [32]:
# SAMPLE DATA GIVEN TO CHECK THE MOST PROFITABLE COUNTRY

countries = ["Canada", "Germany", "France", "USA", "Australia"]

new_orders = pd.DataFrame({
    "Year": [2024, 2027, 2030, 2040, 2050],
    "Order_Month": [7, 4, 8, 1, 9],
    "Order_Quarter": [3]*5,
    "Order_Year": [2024]*5,
    "Customer_Gender": ["Female", "Male", "Female", "Male", "Female"],
    "Customer_Segment": ["Consumer", "New", "Premium", "New", "New"],
    "Region": ["North America", "Europe", "Europe", "North America", "Oceania"],
    "Country": ["Canada", "Germany", "France", "USA", "Australia"],
    "Category": ["Electronics", "Clothing", "Clothing", "Clothing", "Books & Media"],
    "Sub_Category": ["Audio", "Men's Wear", "Books", "Laptops", "Appliances"],
    "Product_Name": ["Wireless Earbuds", "Formal Shirt", "High-Waist Jeans", "Thinking Fast and Slow","Blender Pro"],
    "Unit_Price": [120]*5,
    "Quantity": [200]*5,
    "Discount": [0.10]*5,
    "Shipping_Cost": [20]*5,
    "Shipping_Days": [5]*5,
    "Shipping_Method": ["Express"]*5,
    "Payment_Method": ["Credit Card"]*5,
    "Order_Status": ["Completed"]*5,
    "Season": ["Winter"]*5,
    "scost_per_revenue": [0.02]*5
})

In [33]:
predicted_profit = rf_pipeline.predict(new_orders)

In [34]:
predictions = new_orders.copy()

predictions["Predicted_Profit"] = predicted_profit

In [35]:
predictions.sort_values(
    by="Predicted_Profit",
    ascending=False
)

,Year,Order_Month,Order_Quarter,Order_Year,Customer_Gender,Customer_Segment,Region,Country,Category,Sub_Category,...,Quantity,Discount,Shipping_Cost,Shipping_Days,Shipping_Method,Payment_Method,Order_Status,Season,scost_per_revenue,Predicted_Profit
0,2024,7,3,2024,Female,Consumer,North America,Canada,Electronics,Audio,...,200,0.1,20,5,Express,Credit Card,Completed,Winter,0.02,414.0665
2,2030,8,3,2024,Female,Premium,Europe,France,Clothing,Books,...,200,0.1,20,5,Express,Credit Card,Completed,Winter,0.02,405.8605
1,2027,4,3,2024,Male,New,Europe,Germany,Clothing,Men's Wear,...,200,0.1,20,5,Express,Credit Card,Completed,Winter,0.02,402.0500
3,2040,1,3,2024,Male,New,North America,USA,Clothing,Laptops,...,200,0.1,20,5,Express,Credit Card,Completed,Winter,0.02,401.9465
4,2050,9,3,2024,Female,New,Oceania,Australia,Books & Media,Appliances,...,200,0.1,20,5,Express,Credit Card,Completed,Winter,0.02,391.3945


In [36]:
import joblib

joblib.dump(rf_pipeline, "profit_prediction_model.pkl")

['profit_prediction_model.pkl']

In [38]:
## Tuning random forest regressor hyperparameters to impeove on its default settings

from sklearn.ensemble import RandomForestRegressor

param_grid = {
    'model__n_estimators': [50],
    'model__max_depth': [20],
    'model__min_samples_split': [5],
    'model__min_samples_leaf': [1, 2]
}

random_search = RandomizedSearchCV(
    estimator = rf_pipeline,
    param_distributions = param_grid,
    n_iter = 30,
    cv=5,
    scoring='r2',
    random_state = 34,
    n_jobs=-1
)

random_search.fit(x_train, y_train)

print(random_search.best_params_)
print(random_search.best_score_)


C:\Users\HP\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=30. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


{'model__n_estimators': 50, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__max_depth': 20}
0.8344784334112297


In [39]:
import joblib

joblib.dump(rf_pipeline, "profit_prediction_model.pkl")

['profit_prediction_model.pkl']

*** Note: Having evaluated Linear Regression, Random Forest, and Gradient Boosting and performed hyperparameter tuning using RandomizedSearchCV. The default Random Forest achieved the best performance (R² = 0.8814), so it was selected as the final model ***